In [3]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# Model weights (given)
# --------------------------------------------------
weights = {
    "intercept": -3.82398,
    "age": -0.02990,
    "band_b": -0.09089,
    "band_c": -0.19558,
    "shop_value": 0.02999,
    "shop_frequency": 0.74572
}

# --------------------------------------------------
# Logistic (sigmoid) function
# --------------------------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# --------------------------------------------------
# One-hot encoding for socioeconomic band
# a -> b=0, c=0
# b -> b=1, c=0
# c -> b=0, c=1
# --------------------------------------------------
def encode_band(band):
    if band.lower() == "b":
        return 1, 0
    elif band.lower() == "c":
        return 0, 1
    else:  # band "a"
        return 0, 0

# --------------------------------------------------
# Prediction function
# --------------------------------------------------
def predict_probability(age, band, shop_freq, shop_value):
    b, c = encode_band(band)

    z = (
        weights["intercept"]
        + weights["age"] * age
        + weights["band_b"] * b
        + weights["band_c"] * c
        + weights["shop_value"] * shop_value
        + weights["shop_frequency"] * shop_freq
    )

    return z, sigmoid(z)

# --------------------------------------------------
# Input data (query instances)
# --------------------------------------------------
data = pd.DataFrame({
    "ID": [1, 2, 3, 4, 5],
    "AGE": [56, 21, 48, 37, 32],
    "BAND": ["b", "c", "b", "c", "a"],
    "SHOP_FREQUENCY": [1.60, 4.92, 1.21, 0.72, 1.08],
    "SHOP_VALUE": [109.32, 11.28, 161.19, 170.65, 165.39]
})

# --------------------------------------------------
# Run predictions
# --------------------------------------------------
results = []

for _, row in data.iterrows():
    z, p = predict_probability(
        age=row["AGE"],
        band=row["BAND"],
        shop_freq=row["SHOP_FREQUENCY"],
        shop_value=row["SHOP_VALUE"]
    )

    results.append({
        "ID": row["ID"],
        "Linear score (z)": round(z, 4),
        "Probability": round(p, 4)
    })

results_df = pd.DataFrame(results)
print(results_df)


   ID  Linear score (z)  Probability
0   1           -1.1176       0.2465
1   2           -0.6402       0.3452
2   3            0.3863       0.5954
3   4            0.5289       0.6292
4   5            0.9846       0.7280


In [35]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# Summary statistics from data quality report
# --------------------------------------------------
STATS = {
    "AGE": {"min": 18, "max": 63, "mean": 32.7},
    "SHOP_FREQUENCY": {"min": 0.2, "max": 5.4, "mean": 2.2},
    "SHOP_VALUE": {"min": 5, "max": 230.7, "mean": 101.9}
}

MODE_BAND = "a"

# --------------------------------------------------
# Logistic regression weights (after normalization)
# --------------------------------------------------
weights = {
    "intercept": 0.6679,
    "age": -0.5795,
    "band_b": -0.1981,
    "band_c": -0.2318,
    "shop_value": 3.4091,
    "shop_frequency": 2.0499
}

def range_normalize(x, xmin, xmax):
    return 2 * (x - xmin) / (xmax - xmin) - 1

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def encode_band(band):
    band = str(band).lower()
    if band == "b":
        return 1, 0
    elif band == "c":
        return 0, 1
    else:  # baseline a (or anything imputed to a)
        return 0, 0

# --------------------------------------------------
# Query instances (np.nan for missing is fine)
# --------------------------------------------------
data = pd.DataFrame({
    "ID": [1, 2, 3, 4, 5],
    "AGE": [38, 56, 18, np.nan, 62],
    "BAND": ["a", "b", "c", "b", np.nan],
    "SHOP_FREQUENCY": [1.90, 1.60, 6.00, 1.33, 0.85],
    "SHOP_VALUE": [165.39, 109.32, 10.09, 204.62, 110.50]
})

# Toggle this to match Excel assumption
CLIP_CONTINUOUS_TO_RANGE = True

results = []

for _, row in data.iterrows():

    # --- Impute missing values (use pd.isna for NaN detection) ---
    age = STATS["AGE"]["mean"] if pd.isna(row["AGE"]) else float(row["AGE"])
    band = MODE_BAND if pd.isna(row["BAND"]) else row["BAND"]

    freq = float(row["SHOP_FREQUENCY"])
    value = float(row["SHOP_VALUE"])

    # --- Optional clipping to [min, max] BEFORE normalization ---
    if CLIP_CONTINUOUS_TO_RANGE:
        age = np.clip(age, STATS["AGE"]["min"], STATS["AGE"]["max"])
        freq = np.clip(freq, STATS["SHOP_FREQUENCY"]["min"], STATS["SHOP_FREQUENCY"]["max"])
        value = np.clip(value, STATS["SHOP_VALUE"]["min"], STATS["SHOP_VALUE"]["max"])

    # --- Normalize ---
    age_n = range_normalize(age, STATS["AGE"]["min"], STATS["AGE"]["max"])
    freq_n = range_normalize(freq, STATS["SHOP_FREQUENCY"]["min"], STATS["SHOP_FREQUENCY"]["max"])
    value_n = range_normalize(value, STATS["SHOP_VALUE"]["min"], STATS["SHOP_VALUE"]["max"])

    # --- Encode categorical feature ---
    b, c = encode_band(band)

    # --- Linear score ---
    z = (
        weights["intercept"]
        + weights["age"] * age_n
        + weights["band_b"] * b
        + weights["band_c"] * c
        + weights["shop_value"] * value_n
        + weights["shop_frequency"] * freq_n
    )

    p = sigmoid(z)

    results.append({
        "ID": int(row["ID"]),
        "z": z,
        "Probability": p
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


 ID         z  Probability
  1  1.458850     0.811357
  2 -1.133203     0.243571
  3 -0.189836     0.452683
  4  2.132957     0.894065
  5 -1.645307     0.161744


In [26]:
import numpy as np

# -------------------------
# Sigmoid
# -------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------
# Loss (0/1 version)
# -------------------------
def loss_01(w, x, y):
    z = np.dot(w, x)
    s = sigmoid(z)
    return -y*np.log(s) - (1-y)*np.log(1-s)

# -------------------------
# Analytical gradient
# -------------------------
def grad_01(w, x, y):
    z = np.dot(w, x)
    s = sigmoid(z)
    return x * (s - y)

# -------------------------
# Numerical gradient
# -------------------------
def numerical_grad(f, w, eps=1e-6):
    grad = np.zeros_like(w)
    for i in range(len(w)):
        w_plus = w.copy()
        w_minus = w.copy()
        w_plus[i] += eps
        w_minus[i] -= eps
        grad[i] = (f(w_plus) - f(w_minus)) / (2*eps)
    return grad

# Test
x = np.array([1.0, 2.0, -1.5])
w = np.array([0.3, -0.4, 0.2])
y = 1 # either 0 or 1

analytical = grad_01(w, x, y)
numerical = numerical_grad(lambda w_: loss_01(w_, x, y), w)

print("Analytical:", analytical)
print("Numerical :", numerical)


Analytical: [-0.68997448 -1.37994896  1.03496172]
Numerical : [-0.68997448 -1.37994896  1.03496172]


In [28]:
# -------------------------
# Loss (-1/+1 version)
# -------------------------
def loss_pm1(w, x, y):
    z = np.dot(w, x)
    return np.log(1 + np.exp(-y * z))

# -------------------------
# Analytical gradient
# -------------------------
def grad_pm1(w, x, y):
    z = np.dot(w, x)
    return -y * x * (1 / (1 + np.exp(y * z)))

# Test
x = np.array([1.0, 2.0, -1.5])
w = np.array([0.3, -0.4, 0.2])
y = 1  # try +1 or -1

analytical = grad_pm1(w, x, y)
numerical = numerical_grad(lambda w_: loss_pm1(w_, x, y), w)

print("Analytical:", analytical)
print("Numerical :", numerical)


Analytical: [-0.68997448 -1.37994896  1.03496172]
Numerical : [-0.68997448 -1.37994896  1.03496172]


In [1]:
import numpy as np

# -------------------------
# Sigmoid function
# -------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------
# Model weights
# -------------------------
# [intercept, GL-0, GL-1, GL-2, GL-3, GL-4, GL-5, GL-6, GL-7]
w = np.array([
    0.309,   # intercept
    0.100,
   -0.152,
   -0.163,
    0.191,
   -0.631,
   -0.716,
   -0.478,
   -0.171
])

# -------------------------
# Histogram features for IDs 1, 4, 6
# -------------------------
# Each row: [1 (bias), GL0, GL1, ..., GL7]
X = {
    1: np.array([1, 37, 3, 1, 4, 1, 3, 2, 13]),
    4: np.array([1, 31, 5, 3, 2, 5, 2, 5, 11]),
    6: np.array([1, 31, 3, 5, 2, 3, 6, 2, 12])
}

# -------------------------
# Compute predictions
# -------------------------
for ID, x in X.items():
    z = np.dot(w, x)
    sigma = sigmoid(z)
    print(f"ID {ID}:")
    print(f"   z = {z:.4f}")
    print(f"   sigma(z) = {sigma:.4f}\n")


ID 1:
   z = -1.8040
   sigma(z) = 0.1414

ID 4:
   z = -6.3160
   sigma(z) = 0.0018

ID 6:
   z = -6.6770
   sigma(z) = 0.0013



In [3]:
# -------------------------
# Sigmoid
# -------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------
# Model weights
# -------------------------
w = np.array([
    0.309,   # intercept
    0.100,
   -0.152,
   -0.163,
    0.191,
   -0.631,
   -0.716,
   -0.478,
   -0.171
])

# -------------------------
# Histogram data (bias included)
# -------------------------
X = {
    1: np.array([1, 37, 3, 1, 4, 1, 3, 2, 13]),
    2: np.array([1, 31, 3, 4, 1, 8, 7, 3, 7]),
    3: np.array([1, 38, 2, 3, 0, 1, 1, 5, 14]),
    4: np.array([1, 31, 5, 3, 2, 5, 2, 5, 11]),
    5: np.array([1, 32, 6, 3, 2, 1, 1, 5, 14]),
    6: np.array([1, 31, 3, 5, 2, 3, 6, 2, 12]),
    7: np.array([1, 31, 4, 3, 4, 1, 5, 5, 11]),
    8: np.array([1, 38, 4, 2, 2, 4, 4, 4, 8]),
    9: np.array([1, 38, 3, 2, 3, 4, 4, 1, 9])
}

# -------------------------
# True labels
# -------------------------
y_values = {
    1: 1, 2: 0, 3: 1, 4: 0, 5: 1,
    6: 0, 7: 0, 8: 1, 9: 1
}

# -------------------------
# Function to compute w[j] error
# -------------------------
def wj_error(ID, j):
    x = X[ID]
    y = y_values[ID]
    
    z = np.dot(w, x)
    sigma_z = sigmoid(z)
    
    error = y - sigma_z
    gradient_component = error * sigma_z * (1 - sigma_z) * x[j]
    
    return z, sigma_z, gradient_component

# -------------------------
# Example calculations
# -------------------------

# ID 3, w[0]
z, sigma_z, grad = wj_error(3, 0)
print(f"ID 3, w[0]: {grad:.4f}")

# ID 7, w[2]
z, sigma_z, grad = wj_error(7, 2)
print(f"ID 7, w[2]: {grad:.4f}")

# ID 2, w[8]
z, sigma_z, grad = wj_error(2, 8)
print(f"ID 2, w[8]: {grad:.4f}")



ID 3, w[0]: 0.0503
ID 7, w[2]: -0.0001
ID 2, w[8]: -0.0000


In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# weights: [w0, w(GL0), w(GL1), ..., w(GL7)]
w = np.array([0.309, 0.100, -0.152, -0.163, 0.191, -0.631, -0.716, -0.478, -0.171], float)

# data: (ID, [GL0..GL7], y)
data = [
    (0,[31,3,6,2,7,5,6,4],0),
    (1,[37,3,1,4,1,3,2,13],1),
    (2,[31,3,4,1,8,7,3,7],0),
    (3,[38,2,3,0,1,1,5,14],1),
    (4,[31,5,3,2,5,2,5,11],0),
    (5,[32,6,3,2,1,1,5,14],1),
    (6,[31,3,5,2,3,6,2,12],0),
    (7,[31,4,3,4,1,5,5,11],0),
    (8,[38,4,2,2,2,4,4,8],1),
    (9,[38,3,2,3,4,4,1,9],1),
]

eta = 0.01
update = np.zeros_like(w)

for ID, gl, y in data:
    x = np.array([1] + gl, float)   # x[0]=1 for intercept
    z = w @ x
    s = sigmoid(z)

    # "w[j] error" as defined in your prompt: (y - s) * s*(1-s) * x[j]
    update += (y - s) * s * (1 - s) * x

w_new = w + eta * update

print("Sum update:", update)
print("New weights:", w_new)


Sum update: [0.22626333 8.34835953 0.72124777 0.42285397 0.58780243 0.32872285
 0.57708365 0.68868678 2.80609595]
New weights: [ 0.31126263  0.1834836  -0.14478752 -0.15877146  0.19687802 -0.62771277
 -0.71022916 -0.47111313 -0.14293904]


In [8]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------
# Initial weights
# -------------------------
w = np.array([0.309, 0.100, -0.152, -0.163, 0.191,
              -0.631, -0.716, -0.478, -0.171], dtype=float)

# -------------------------
# Dataset (GL0..GL7, y)
# -------------------------
data = [
    (0,[31,3,6,2,7,5,6,4],0),
    (1,[37,3,1,4,1,3,2,13],1),
    (2,[31,3,4,1,8,7,3,7],0),
    (3,[38,2,3,0,1,1,5,14],1),
    (4,[31,5,3,2,5,2,5,11],0),
    (5,[32,6,3,2,1,1,5,14],1),
    (6,[31,3,5,2,3,6,2,12],0),
    (7,[31,4,3,4,1,5,5,11],0),
    (8,[38,4,2,2,2,4,4,8],1),
    (9,[38,3,2,3,4,4,1,9],1),
]

learning_rate = 0.01
total_update = 0.0

print("Individual contributions to w[1] (GL-0 weight):\n")

for ID, gl, y in data:
    x = np.array([1] + gl, dtype=float)
    z = np.dot(w, x)
    p = sigmoid(z)

    error = y - p
    slope = p * (1 - p)

    # weight 1 corresponds to GL-0 → x[1]
    contribution = error * slope * x[1]

    total_update += contribution

    print(f"ID {ID}: contribution = {contribution:.6f}")

print("\nTotal summed update for w[1]:", round(total_update,6))

# Update weight 1
w_new_1 = w[1] + learning_rate * total_update

print("\nOld w[1]:", w[1])
print("New w[1]:", round(w_new_1,6))


Individual contributions to w[1] (GL-0 weight):

ID 0: contribution = -0.000000
ID 1: contribution = 3.856208
ID 2: contribution = -0.000000
ID 3: contribution = 1.911807
ID 4: contribution = -0.000101
ID 5: contribution = 0.776523
ID 6: contribution = -0.000049
ID 7: contribution = -0.000617
ID 8: contribution = 0.759841
ID 9: contribution = 1.044746

Total summed update for w[1]: 8.34836

Old w[1]: 0.1
New w[1]: 0.183484


In [ ]:
import numpy as np

# -------------------------
# Sigmoid
# -------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------
# New weights (after LSE update)
# -------------------------
# [intercept, GL0, GL1, GL2, GL3, GL4, GL5, GL6, GL7]
w_new = np.array([
    0.311263,
    0.183484,
   -0.144788,
   -0.158771,
    0.196878,
   -0.627713,
   -0.710229,
   -0.471113,
   -0.142939
])

# -------------------------
# New instances (GL0..GL7)
# -------------------------
instances = [
    [35, 1, 5, 4, 5, 2, 4, 8],
    [30, 6, 2, 0, 5, 4, 4, 13]
]

# -------------------------
# Compute predictions
# -------------------------
for i, gl in enumerate(instances, start=1):
    x = np.array([1] + gl, dtype=float)  # add intercept term
    z = np.dot(w_new, x)
    prob = sigmoid(z)

    print(f"Instance {i}")
    print(f"   z = {z:.4f}")
    print(f"   sigma(z) = {prob:.4f}")

    predicted_class = 1 if prob >= 0.0 else 0
    print(f"   Predicted class = {predicted_class}\n")


Instance 1
   z = -1.0049
   sigma(z) = 0.2680
   Predicted class = 0

Instance 2
   z = -5.0926
   sigma(z) = 0.0061
   Predicted class = 0



In [10]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

eta = 0.01  # learning rate

# original weights
w = np.array([0.309, 0.100, -0.152, -0.163, 0.191, -0.631, -0.716, -0.478, -0.171], float)

# ID 0: x = [1, GL0..GL7], y = 0
x = np.array([1, 31, 3, 6, 2, 7, 5, 6, 4], float)
y = 0.0

z = w @ x
p = sigmoid(z)

grad = (p - y) * x              # MLE gradient for one sample
w_new = w - eta * grad          # gradient descent on cross-entropy

print("z =", z)
print("sigma(z) =", p)
print("grad =", grad)
print("w_new =", w_new)


z = -9.191999999999998
sigma(z) = 0.00010184058639056558
grad = [0.00010184 0.00315706 0.00030552 0.00061104 0.00020368 0.00071288
 0.0005092  0.00061104 0.00040736]
w_new = [ 0.30899898  0.09996843 -0.15200306 -0.16300611  0.19099796 -0.63100713
 -0.71600509 -0.47800611 -0.17100407]


In [11]:
import numpy as np

# --------------------------------------------------
# Sigmoid
# --------------------------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# --------------------------------------------------
# Original weights
# --------------------------------------------------
w_original = np.array([
    0.309,   # intercept
    0.100,
   -0.152,
   -0.163,
    0.191,
   -0.631,
   -0.716,
   -0.478,
   -0.171
], dtype=float)

# --------------------------------------------------
# Training dataset (ID, GL0..GL7, y)
# --------------------------------------------------
data = [
    (0,[31,3,6,2,7,5,6,4],0),
    (1,[37,3,1,4,1,3,2,13],1),
    (2,[31,3,4,1,8,7,3,7],0),
    (3,[38,2,3,0,1,1,5,14],1),
    (4,[31,5,3,2,5,2,5,11],0),
    (5,[32,6,3,2,1,1,5,14],1),
    (6,[31,3,5,2,3,6,2,12],0),
    (7,[31,4,3,4,1,5,5,11],0),
    (8,[38,4,2,2,2,4,4,8],1),
    (9,[38,3,2,3,4,4,1,9],1),
]

eta = 0.01
M = len(data)

# --------------------------------------------------
# Compute batch MLE gradient
# --------------------------------------------------
gradient_sum = np.zeros_like(w_original)

for ID, gl, y in data:
    x = np.array([1] + gl, dtype=float)
    z = np.dot(w_original, x)
    p = sigmoid(z)

    gradient_sum += (p - y) * x   # MLE gradient

# SUM update
w_sum_update = w_original - eta * gradient_sum

# MEAN update
w_mean_update = w_original - eta * (gradient_sum / M)

print("Original weights:\n", w_original)
print("\nGradient (SUM):\n", gradient_sum)
print("\nWeights after SUM update:\n", w_sum_update)
print("\nWeights after MEAN update:\n", w_mean_update)

# --------------------------------------------------
# Evaluate new instances
# --------------------------------------------------
new_instances = [
    [35,1,5,4,5,2,4,8],
    [30,6,2,0,5,4,4,13]
]

print("\n--- Predictions on new instances ---\n")

for idx, gl in enumerate(new_instances, start=1):
    x = np.array([1] + gl, dtype=float)

    # SUM update prediction
    z_sum = np.dot(w_sum_update, x)
    p_sum = sigmoid(z_sum)

    # MEAN update prediction
    z_mean = np.dot(w_mean_update, x)
    p_mean = sigmoid(z_mean)

    print(f"Instance {idx}")
    print(f"  SUM update:  z = {z_sum:.4f}, sigma(z) = {p_sum:.4f}")
    print(f"  MEAN update: z = {z_mean:.4f}, sigma(z) = {p_mean:.4f}\n")


Original weights:
 [ 0.309  0.1   -0.152 -0.163  0.191 -0.631 -0.716 -0.478 -0.171]

Gradient (SUM):
 [  -4.71886285 -172.6651813   -17.10750585  -10.48648664  -10.22996186
   -8.59989218  -12.25944968  -16.15969322  -54.49905177]

Weights after SUM update:
 [ 0.35618863  1.82665181  0.01907506 -0.05813513  0.29329962 -0.54500108
 -0.5934055  -0.31640307  0.37399052]

Weights after MEAN update:
 [ 0.31371886  0.27266518 -0.13489249 -0.15251351  0.20122996 -0.62240011
 -0.70374055 -0.46184031 -0.11650095]

--- Predictions on new instances ---

Instance 1
  SUM update:  z = 63.0051, sigma(z) = 1.0000
  MEAN update: z = 2.4656, sigma(z) = 0.9217

Instance 2
  SUM update:  z = 53.6516, sigma(z) = 1.0000
  MEAN update: z = -1.9095, sigma(z) = 0.1290

